# Lecture : Shallow Graph Learning

## Lab 01 : Node Classification on SBM Graphs with Graphlet Degree Vectors -- Exercise

### Xavier Bresson, Ryoji Kubo



The main steps of this notebook are:

1. Generate a dataset of SBM graphs with **5 clusters** and graph sizes sampled uniformly from **[140, 160]**.
1. Compute a node-level **Graphlet Degree Vector (GDV)** for each node.
1. Train a **linear SVM** supervised node classifier using the GDVs as features.
1. Evaluate the classifier on **new, unseen 5-cluster SBM graphs**.

The training, validation, and test sets are split **at the graph level rather than the node level**.  
This means that every node in the test set comes from a graph that the model has **never seen during training**.


## Graphlet Degree Vector

We use the **15 automorphism orbits** associated with connected **graphlets on 2–4 nodes**:

- Orbit 0: edge
- Orbits 1–3: roles in the 3-node path and triangle
- Orbits 4–14: roles in the connected 4-node graphlets

<div align="center">
  <img src="pic/orbits.png" width="50%">
</div>

For a node $v$, the GDV is defined as

$$
\mathbf h_v =
[h_{v,0},h_{v,1},\ldots,h_{v,14}]^\top\in\mathbb{N}^{15},
$$

where $h_{v,r}$ counts how many times node $v$ participates in the graphlet orbit $r$.

The first four coordinates are computed exactly. But the 4-node orbit counts are estimated by rooted Monte Carlo sampling to keep the notebook fast.

Because graph sizes vary, we also use a graph-size-normalized GDV for classification.


In [ ]:
# For Google Colaboratory
import sys, os
if 'google.colab' in sys.modules:
    # mount google drive
    from google.colab import drive
    drive.mount('/content/gdrive')
    path_to_file = '/content/gdrive/My Drive/CS5284_2026_codes/05_Shallow_Learning'
    print(path_to_file)
    # change current path to the folder containing "path_to_file"
    os.chdir(path_to_file)
    !pwd
    !pip -q install -f https://data.dgl.ai/wheels/torch-2.6/repo.html dgl==2.5.0 # Install DGL
    

In [ ]:
import math
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay


## Step 1 : Generate 5-cluster SBM graphs

A stochastic block model (SBM) divides nodes into clusters. Two nodes are connected with probability $p_{\mathrm{in}}$ if they belong to the same cluster, and with probability $p_{\mathrm{out}}$ if they belong to different clusters:

$$
A_{ij}\sim \mathrm{Bernoulli}(P_{z_i z_j}),
$$

where $z_i$ is the cluster label of node $i$, $P_{z_i z_j}=p_{\mathrm{in},z_i}$ when $z_i=z_j$, and $P_{z_i z_j}=p_{\mathrm{out}}$ when $z_i\neq z_j$.



If all five clusters used the same probabilities, their labels would be interchangeable. A local, unlabeled feature such as a GDV could not consistently tell "cluster 0" from "cluster 1" across different graphs.

To give each cluster a consistent structural role, we use different within-cluster probabilities:

$$
p_{\mathrm{in},c} = 0.35 + 0.15 \cdot c, \ c\in\{0,1,2,3,4\} 
\qquad
p_{\mathrm{out},c}=0.01,\ \forall c.
$$

Thus, cluster 0 is always the sparsest community, while cluster 4 is always the densest.


In [ ]:
# Create balanced block sizes with small random variation
def balanced_block_sizes(n, k):
    base, rem = divmod(n, k) # n=153, k=5, base=30, rem=3
    sizes = np.array( [base + (1 if i < rem else 0) for i in range(k)] ) # sizes=[31 31 31 30 30]
    # remove/add 2 nodes randomly
    for _ in range(2*k):
        a, b = np.random.choice(k, size=2, replace=False) # a, b = 4 3
        sizes[a] -= 2
        sizes[b] += 2
    sizes = sizes.tolist() # sizes = [29 31 35 28 30]
    return sizes

# We use different within-cluster probabilities for the SBM probability matrix
# P = [[0.35 0.01 0.01 0.01 0.01]
#      [0.01 0.5  0.01 0.01 0.01]
#      [0.01 0.01 0.65 0.01 0.01]
#      [0.01 0.01 0.01 0.8  0.01]
#      [0.01 0.01 0.01 0.01 0.95]]
def sbm_probability_matrix(k=5, p_in_min=0.35, p_in_max=0.95, p_out=0.01):
    p_in = np.linspace(p_in_min, p_in_max, k) # p_in = [0.35 0.5  0.65 0.8  0.95]in)
    P = np.full((k, k), p_out, dtype=float)
    np.fill_diagonal(P, p_in)
    return P

# SBM
def generate_sbm_graph(n, k=5):
    sizes = balanced_block_sizes(n, k) # sizes = [30, 28, 36, 32, 33]
    P = sbm_probability_matrix(k=k)
    G = nx.stochastic_block_model(sizes, P) # G = 'stochastic_block_model' with 159 nodes and 1752 edges
    y = np.concatenate([ np.full(size, block_id, dtype=int) for block_id, size in enumerate(sizes) ])
    # node labels : y = [0 .. 0 1 .. 1 2 .. 2 3 .. 3 4 .. 4]
    return G, y, sizes, P

# Plot the graph in projected 2D space
def plot_graph_blocks(G, labels):
    pos = nx.spring_layout(G)
    plt.figure(figsize=(7, 6))
    nx.draw_networkx_edges(G, pos, alpha=0.25, width=0.7)
    nx.draw_networkx_nodes(G, pos, node_color=labels, cmap="tab10", node_size=55)
    plt.axis("off")
    plt.show()
    
# Sample graph size uniformly from [140, 160]
n = np.random.randint(140, 161)
print("Number of nodes:", n)
G, y, sizes, P = generate_sbm_graph(n=n, k=5)
print("Block sizes:", sizes)
print("SBM probability matrix:", np.round(P, 3))
print("Node labels:", y[:10])

# Plot adjacency graph
plt.figure(1)
A = nx.to_numpy_array(G)
plt.spy(A, precision=0.01, markersize=1)
plt.show()

# Plot graph in 2D space
plot_graph_blocks(G, y)


## Step 2 : Compute the 15-dimensional GDV

For graphlets with up to 3 nodes, the orbit counts can be computed exactly.

For a root node $v$:

* **Orbit 0**: edge
* **Orbit 1**: endpoint of an induced 3-node path
* **Orbit 2**: center of an induced 3-node path
* **Orbit 3**: triangle

For 4-node graphlets, the notebook samples triples of other nodes uniformly from the $\binom{n-1}{3}=\frac{(n-1)(n-2)(n-3)}{6}=O(n^3)$ possible rooted 4-node subsets. For (n=150), $\binom{n-1}{3}=\binom{149}{3}=540,274$.

For each sampled subset, the induced 4-node graph determines the orbit occupied by the root $v$, provided the induced graph is connected.

This also makes it clearer why the GDV has 15 dimensions: the connected graphlets on 2, 3, and 4 nodes contain **15 distinct node orbits** in total.


In [ ]:
ORBIT_NAMES = [
    "0: edge",
    "1: P3 endpoint",
    "2: P3 center",
    "3: triangle",
    "4: P4 endpoint",
    "5: P4 internal",
    "6: 3-star leaf",
    "7: 3-star center",
    "8: 4-cycle",
    "9: tailed-triangle tail",
    "10: tailed-triangle degree-2",
    "11: tailed-triangle attachment",
    "12: diamond degree-2",
    "13: diamond degree-3",
    "14: K4",
]

def is_connected_small(B):
    """Connectivity test for a small adjacency matrix."""
    k = B.shape[0]
    seen = {0}
    stack = [0]

    while stack:
        u = stack.pop()
        for v in np.flatnonzero(B[u]):
            v = int(v)
            if v not in seen:
                seen.add(v)
                stack.append(v)

    return len(seen) == k

def orbit_of_root_in_4node_graph(B, root=0):
    """
    Return orbit ID 4,...,14 for the root node.
    Return -1 if the induced 4-node subgraph is disconnected.
    """
    if not is_connected_small(B):
        return -1

    d = B.sum(axis=1).astype(int)
    seq = tuple(sorted(d.tolist()))
    dr = int(d[root])

    # P4
    if seq == (1, 1, 2, 2):
        return 4 if dr == 1 else 5

    # K1,3
    if seq == (1, 1, 1, 3):
        return 6 if dr == 1 else 7

    # C4
    if seq == (2, 2, 2, 2):
        return 8

    # Tailed triangle
    if seq == (1, 2, 2, 3):
        if dr == 1:
            return 9
        if dr == 2:
            return 10
        return 11

    # Diamond = K4 minus one edge
    if seq == (2, 2, 3, 3):
        return 12 if dr == 2 else 13

    # K4
    if seq == (3, 3, 3, 3):
        return 14

    raise RuntimeError(
        f"Unexpected connected 4-node degree sequence: {seq}"
    )

def normalize_gdv_for_graph_size(X, n):
    """
    Normalize orbit counts by the number of possible rooted subsets.
    This reduces variation caused solely by graph size.
    """
    Z = X.astype(float).copy()
    Z[:, 0] /= max(n - 1, 1)
    Z[:, 1:4] /= max(math.comb(n - 1, 2), 1)
    Z[:, 4:15] /= max(math.comb(n - 1, 3), 1)
    return Z

In [ ]:
# Plot the graplet orbits
from lib.utils import plot_all_orbits
plot_all_orbits(ORBIT_NAMES)


## Exercise 1: Implement the orbit counts for graphlets upto 3 nodes

Given a graph, compute the graphlet orbit counts $O_3(v)$, $O_2(v)$, and $O_1(v)$ for all nodes $v$ based on the following mathematical formulas. We will also be using $C$, the common neighbor matrix. $C_{vu}$ represents the number of common neighbors between node $v$ and node $u$. This can be computed by squaring the adjacency matrix: $C = A^2$.

### Question 1.1 Orbit 3 (Triangles)
The number of triangles node $v$ is a part of is denoted as $O_3(v)$.

Mathematically, a triangle exists if $v$ is connected to $u$, and they share a common neighbor. If we sum the common neighbors over all of $v$'s direct connections, we will count every triangle exactly twice (once from the perspective of each of the other two nodes in the triangle).

$$O_3(v) = \frac{1}{2} \sum_{u=1}^{n} \left( A_{vu} \cdot C_{vu} \right)$$

### Question 1.2 Orbit 2 (Center of induced P3)
An induced path of 3 nodes (P3) centered at $v$ means $v$ connects to two nodes that are not connected to each other. The number of such configurations is denoted as $O_2(v)$.

Mathematically, this is the total number of pairs of neighbors of $v$, minus the pairs that are connected to each other (which form triangles, $O_3(v)$).

$$O_2(v) = \binom{d(v)}{2} - O_3(v) = \frac{d(v)(d(v) - 1)}{2} - O_3(v)$$

### Question 1.3 Orbit 1 (Endpoint of induced P3)
The number of times node $v$ acts as the starting or ending point of an induced 3-node path is denoted as $O_1(v)$.

Mathematically, for every neighbor $u \in N(v)$, we want to count how many connections $u$ has that are not $v$, and are not already connected to $v$ (since that would form a triangle).

$$O_1(v) = \sum_{u \in N(v)} \Big( d(u) - 1 - C_{vu} \Big)$$

Constraints & Hints:

Try to use vectorized NumPy operations for $O_3$ and $O_2$ rather than standard for loops to optimize performance. Element-wise multiplication (*) and dot products (@) will be your friends. For $O_1$, a standard loop iterating through the nodes for v in range(n): is acceptable, provided you use NumPy arrays to efficiently sum over the neighbors inside the loop.

In [ ]:
def graphlet_degree_vectors(G):
    """
    Compute a 15-dimensional GDV for every node.
     Orbits 0-3: exact.
     Orbits 4-14: Monte Carlo estimates using uniformly sampled rooted 4-node subsets.
    """

    n = G.number_of_nodes()
    nodes = np.arange(n)

    A = nx.to_numpy_array(G, nodelist=nodes, dtype=np.int8)
    np.fill_diagonal(A, 0)

    degree = A.sum(axis=1).astype(float)

    # Common-neighbor counts
    common = A @ A

    ###############################################
    # YOUR CODE START
    ###############################################
    # Orbit 3: triangle. Compute O_3 using the equation above.
    triangles = 

    # Orbit 2: center of induced P3. Compute O_2 using the equation above.
    p3_center = 

    # Orbit 1: endpoint of induced P3. We will compute this for each node v
    p3_endpoint = np.zeros(n, dtype=float)
    for v in range(n):
        nbrs = np.flatnonzero(A[v])
        if len(nbrs):
            # Compute O_1(v) using the equation above.
            p3_endpoint[v] = 
    ###############################################
    # YOUR CODE END
    ###############################################

    X = np.zeros((n, 15), dtype=float)
    X[:, 0] = degree
    X[:, 1] = p3_endpoint
    X[:, 2] = p3_center
    X[:, 3] = triangles

    # Sample rooted 4-node subsets.
    total_rooted_subsets = math.comb(n - 1, 3)
    min_n_samples_4_per_node = 500
    M = min(min_n_samples_4_per_node, total_rooted_subsets)
    for v in range(n):
        others = np.delete(nodes, v)
        for _ in range(M):
            rest = np.random.choice(others, size=3, replace=False)
            idx = np.array([v, *rest], dtype=int)
            B = A[np.ix_(idx, idx)]
            orbit = orbit_of_root_in_4node_graph(B, root=0)
            if orbit >= 0:
                X[v, orbit] += 1.0

    # Scale sampled counts to estimated total rooted counts
    if M > 0:
        X[:, 4:15] *= total_rooted_subsets / M

    return X

In [ ]:
# Compute 15-dim GDV for every node 
X_raw = graphlet_degree_vectors(G)
print("Raw GDV shape:", X_raw.shape)

# Compute 15-dim GDV frequency for every node 
X = normalize_gdv_for_graph_size(X_raw, G.number_of_nodes())
print("Normalized GDV shape:", X.shape)

# Print 3 random 15-dim GDV nodes
nodes = np.random.choice(len(y), size=3, replace=False)
for node in nodes:
    print(f"\nNode {node}, true cluster {y[node]}")
    for name, value in zip(ORBIT_NAMES, X_raw[node]):
        if value > 0:
            print(f"  {name:32s} {value:10.2f}")


## Step 3 : Generate train, validation and test sets

All three splits consist of 5-cluster graphs, but the test graphs are generated independently and are never used during training.


In [ ]:
N_TRAIN_GRAPHS = 30 
N_TEST_GRAPHS = 10

def build_dataset(num_graphs, k=5):
    dataset = []
    for graph_id in range(num_graphs):
        # Sample graph size uniformly from [140, 160]
        n = np.random.randint(140, 161)
        G, y, sizes, P = generate_sbm_graph(n=n, k=k)
        gdv_raw = graphlet_degree_vectors(G)
        gdv = normalize_gdv_for_graph_size(gdv_raw, n)
        dataset.append({ "G": G, "y": y, "sizes": sizes, "P": P, "gdv": gdv })
    return dataset

train_set = build_dataset(N_TRAIN_GRAPHS, k=5)
test_set = build_dataset(N_TEST_GRAPHS, k=5)

print("Training graphs:", len(train_set))
print("Test graphs:", len(test_set))

print("Training graph node counts:", [d["G"].number_of_nodes() for d in train_set])
print("Test graph node counts:", [d["G"].number_of_nodes() for d in test_set])


## Step 4 : Train a 5-class node classifier from GDVs with linear SVM

Each node becomes one training sample that is mapped to one of the five classes:

$$
\mathbf h_v\in\mathbb{N}^{15}
\longrightarrow
y_v\in\{0,1,2,3,4\}.
$$

We will use:

1. Standardized GDV coordinates,
1. Multinomial logistic regression (SVM),
1. Train the linear model with sklearn.

The model is then refit on all train graphs and evaluated on new unseen test graphs.


In [ ]:
# Prepare (X,y), where X in R^{N x 15} is the feature vector and y in {0,1,2,3,4}^N is the label vector
def stack_node_dataset(dataset):
    X = np.vstack([d["gdv"] for d in dataset])
    y = np.concatenate([d["y"] for d in dataset])
    return X, y

X_train, y_train = stack_node_dataset(train_set)
X_test, y_test = stack_node_dataset(test_set)

print("Train nodes:", X_train.shape)   
print("Test nodes:", X_test.shape)


## Step 5 : Final evaluation on unseen 5-cluster graphs


In [ ]:
final_model = Pipeline([ ("scaler", StandardScaler()), ("classifier", LogisticRegression(C=100, max_iter=4000)) ])
final_model.fit( X_train, y_train )
pred_test = final_model.predict(X_test)
print("Test accuracy:", accuracy_score(y_test, pred_test))


In [ ]:
# Print the confusion matrix
# A confusion matrix compares the true class labels with the labels predicted by a classifier
# For K classes, the confusion matrix C ∈ N^K×K is defined as 
#  C_ij = # {v : y_v=i, y_v^pred=j}, where y_v is the true class of node v and y_v^pred is its predicted class.
cm = confusion_matrix(y_test, pred_test, labels=np.arange(5))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm, display_labels=["cluster 0", "cluster 1", "cluster 2", "cluster 3", "cluster 4"])
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
plt.title("Node classification on unseen 5-cluster SBM graphs")
plt.show()

# Visualize true and predicted labels for one unseen test graph
example_id = 0
d = test_set[example_id]
pred = final_model.predict(d["gdv"])
pos = nx.spring_layout(d["G"])

plt.figure(figsize=(7, 6))
nx.draw_networkx_edges(d["G"], pos, alpha=0.22, width=0.7)
nx.draw_networkx_nodes(d["G"], pos, node_color=d["y"], cmap="tab10", node_size=55)
plt.title("Unseen test graph: true node classes")
plt.axis("off")
plt.show()

plt.figure(figsize=(7, 6)) 
nx.draw_networkx_edges(d["G"], pos, alpha=0.22, width=0.7)
nx.draw_networkx_nodes(d["G"],pos,node_color=pred,cmap="tab10",node_size=55)
plt.title("Unseen test graph: GDV-predicted node classes")
plt.axis("off")
plt.show()


## Step 7 : Does the full GDV help beyond degree?

Orbit 0 is simply node degree.

As a useful baseline, compare "degree only" vs. "15-dimensional GDV".

If the GDV performs better, the higher-order local topology is contributing useful information beyond ordinary degree.


In [ ]:
# Degree-only baseline using orbit 0
def stack_node_dataset_slice(dataset, feature_slice):
    X = np.vstack([ d["gdv"][:, feature_slice] for d in dataset ])
    y = np.concatenate([ d["y"] for d in dataset ])
    return X, y
    
X_train_degree, y_train_degree = stack_node_dataset_slice(train_set, feature_slice=[0])
X_test_degree, y_test_degree = stack_node_dataset_slice(test_set, feature_slice=[0])

degree_model = Pipeline([("scaler", StandardScaler()),("classifier", LogisticRegression(C=100, max_iter=4000))])
degree_model.fit(X_train_degree, y_train_degree)
pred_degree = degree_model.predict(X_test_degree)

print("Degree-only accuracy:", accuracy_score(y_test_degree, pred_degree) )
print("Full GDV accuracy:", accuracy_score(y_test, pred_test) )


## Limitation

The cluster labels are learnable here because the five blocks have consistent structural identities across graphs through their different within-block probabilities.

If **all five blocks use identical SBM parameters**, then the block IDs are arbitrary permutations. In that setting, node classification by fixed labels is not well-defined from GDVs alone.

What would you suggest to solve the node classification task then?
